# DiT Architecture: Assembling the Full Model

In this notebook we explore the **Diffusion Transformer (DiT)** -- the neural network at the heart of modern video generation systems like Wan 2.1.  We will instantiate our miniature **NanoDiT** model, trace data through every layer, count parameters component-by-component, and see exactly how the same architecture scales from ~1.5 M parameters (Nano) to ~14 B parameters (Wan 14B).

## What is a Diffusion Transformer?

A Diffusion Transformer (DiT) replaces the traditional U-Net backbone used in earlier diffusion models (DDPM, Stable Diffusion v1) with a pure Transformer architecture.  The key differences from a *standard* language-model Transformer are:

| Aspect | Standard LLM Transformer | Diffusion Transformer (DiT) |
|--------|--------------------------|-----------------------------|
| Input | Token embeddings | Patchified noisy latents |
| Self-attention mask | **Causal** (lower-triangular) | **Bidirectional** (no mask) -- every patch can see every other patch |
| Conditioning | None or prefix tokens | **Timestep modulation** via AdaIN (shift/scale/gate from $t$) |
| Cross-attention | None (or encoder-decoder) | **Text conditioning** -- video patches attend to text embeddings |
| Positional encoding | 1-D (token position) | **3-D RoPE** (temporal + height + width) |
| Output | Next-token logits | Predicted velocity $v = \epsilon - x_0$ |

The **modulation** mechanism is perhaps the most distinctive feature: at every layer, the timestep embedding is projected into 6 parameters (shift, scale, and gate for both self-attention and FFN sub-layers).  This lets the model behave differently at different noise levels -- it needs coarse-grained understanding at high noise and fine-grained detail at low noise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import torch.nn as nn
from nano_video_gen.model.nano_dit import NanoDiT

# ---- Instantiate the NanoDiT with default (nano-scale) parameters ----
model = NanoDiT(
    dim=128,
    in_dim=4,
    ffn_dim=512,
    out_dim=4,
    text_dim=64,
    freq_dim=64,
    num_heads=4,
    num_layers=2,
    patch_size=(1, 2, 2),
)
model.eval()

print("=" * 70)
print("NanoDiT Model Architecture")
print("=" * 70)
print(model)

## Parameter Breakdown

Let us count the learnable parameters in every named component of the model.  This gives insight into *where* the capacity of the network lives.

In [ ]:
def count_params(module: nn.Module) -> int:
    """Count all learnable parameters in a module."""
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


# ---- Component-level counts ----
components = {
    "patch_embedding": model.patch_embedding,
    "text_embedding":  model.text_embedding,
    "time_embedding":  model.time_embedding,
    "time_projection": model.time_projection,
}

# Add each block separately
for i, block in enumerate(model.blocks):
    components[f"blocks[{i}]"] = block

components["head"] = model.head

# ---- Print formatted table ----
total = count_params(model)
print(f"{'Component':<22} {'Parameters':>12}  {'% of Total':>10}")
print("-" * 48)
running = 0
for name, mod in components.items():
    n = count_params(mod)
    running += n
    print(f"{name:<22} {n:>12,}  {n / total * 100:>9.1f}%")
print("-" * 48)
print(f"{'TOTAL':<22} {total:>12,}  {total / total * 100:>9.1f}%")

# Sanity-check: the running sum should match total (buffers like freqs are not learnable)
if running != total:
    print(f"\nNote: {total - running:,} params are in buffers or other sub-modules.")

## Data Flow Through the Model

Here is the step-by-step journey of a tensor through NanoDiT:

1. **Input**: Noisy latent video $x_t \in \mathbb{R}^{B \times C \times T \times H \times W}$, e.g. `[1, 4, 4, 16, 16]`.
2. **Time embedding**: Scalar timestep $t \xrightarrow{\text{sinusoidal}} \mathbb{R}^{\text{freq\_dim}} \xrightarrow{\text{MLP}} \mathbb{R}^{\text{dim}}$.
3. **Time projection**: $\mathbb{R}^{\text{dim}} \xrightarrow{\text{SiLU + Linear}} \mathbb{R}^{6 \times \text{dim}}$ (6 modulation parameters per block).
4. **Text embedding**: Text features $\mathbb{R}^{B \times S_{\text{text}} \times \text{text\_dim}} \xrightarrow{\text{MLP}} \mathbb{R}^{B \times S_{\text{text}} \times \text{dim}}$.
5. **Patchify**: $x_t \xrightarrow{\text{Conv3d}} \mathbb{R}^{B \times \text{dim} \times f \times h \times w} \xrightarrow{\text{reshape}} \mathbb{R}^{B \times (fhw) \times \text{dim}}$.
6. **3D RoPE**: Precomputed rotation frequencies for each (frame, height, width) position.
7. **Transformer blocks** (repeated $L$ times):
   - AdaIN modulation $\rightarrow$ Self-attention (with RoPE) $\rightarrow$ gated residual
   - Cross-attention with text embeddings $\rightarrow$ residual
   - AdaIN modulation $\rightarrow$ FFN $\rightarrow$ gated residual
8. **Head**: LayerNorm + modulation + Linear projection to $\text{out\_dim} \times \prod(\text{patch\_size})$.
9. **Unpatchify**: Rearrange back to $\mathbb{R}^{B \times C_{\text{out}} \times T \times H \times W}$.

In [ ]:
# ---- Create dummy inputs matching the nano model dimensions ----
B = 1
x = torch.randn(B, 4, 4, 16, 16)       # [B, in_dim, T, H, W]
timestep = torch.tensor([500.0])         # [B]
context = torch.randn(B, 8, 64)          # [B, seq_len, text_dim]

print("Input shapes:")
print(f"  x (noisy latent):  {list(x.shape)}")
print(f"  timestep:          {list(timestep.shape)}  value={timestep.item():.0f}")
print(f"  context (text):    {list(context.shape)}")

# ---- Forward pass ----
with torch.no_grad():
    output = model(x, timestep, context)

print(f"\nOutput shape: {list(output.shape)}")
print(f"  -> Same spatial dimensions as input: {list(x.shape) == list(output.shape)}")
print(f"  -> Output statistics: mean={output.mean().item():.4f}, std={output.std().item():.4f}")

In [ ]:
# ---- Visualize data flow through the model ----
from nano_video_gen.visualization.viz import visualize_data_flow
import matplotlib.pyplot as plt

fig = visualize_data_flow(model, x, timestep, context, figsize=(16, 5))
plt.show()

## Architecture Comparison: Wan 14B vs Wan 1.3B vs Nano

Our NanoDiT preserves **every layer type** present in the full Wan 14B model.  The only difference is the width (dim), depth (layers), and number of attention heads.  Here is the full comparison:

| Component | Wan 14B | Wan 1.3B | Nano |
|-----------|--------:|---------:|-----:|
| `dim` (model width) | 5120 | 1536 | 128 |
| `num_heads` | 40 | 12 | 4 |
| `head_dim` (= dim / heads) | 128 | 128 | 32 |
| `num_layers` (depth) | 40 | 30 | 2 |
| `ffn_dim` | 13824 | 8960 | 512 |
| `in_dim` (latent channels) | 16 | 16 | 4 |
| `text_dim` (text encoder dim) | 4096 | 4096 | 64 |
| `freq_dim` (timestep emb dim) | 256 | 256 | 64 |
| `patch_size` | [1, 2, 2] | [1, 2, 2] | [1, 2, 2] |
| **~Total parameters** | **~14 B** | **~1.3 B** | **~1.5 M** |

Notice the structural invariants that are shared across all three:
- The same `patch_size = [1, 2, 2]` (no temporal patching, 2x spatial)
- The same layer ordering: self-attn -> cross-attn -> FFN in every block
- The same 6-parameter AdaIN modulation per block (shift/scale/gate for MSA and FFN)
- The same Q/K RMSNorm before attention
- The same 3D RoPE for positional encoding
- The same output head structure with its own modulation

## What Does "Scaling Up" Mean?

Scaling a DiT model means changing just a few hyperparameters while keeping the architecture **exactly the same**:

1. **Wider** -- increase `dim` (e.g., 128 $\rightarrow$ 5120): every linear layer, every embedding, every norm grows proportionally.  Since linear layers have $O(\text{dim}^2)$ parameters, doubling the width roughly quadruples the parameter count.

2. **Deeper** -- increase `num_layers` (e.g., 2 $\rightarrow$ 40): more transformer blocks means more sequential computation.  Parameters grow linearly with depth.

3. **More heads** -- increase `num_heads` (e.g., 4 $\rightarrow$ 40): this does not change the parameter count (Q, K, V projections have the same shape) but allows the model to attend to more diverse patterns simultaneously.

The NanoDiT has **every layer type that Wan 14B has** -- it is a faithful miniature.  When you understand how data flows through NanoDiT, you understand how data flows through the full 14B-parameter model.

In [ ]:
import math

def estimate_params(
    dim: int, ffn_dim: int, num_layers: int, num_heads: int,
    in_dim: int, out_dim: int, text_dim: int, freq_dim: int,
    patch_size: tuple = (1, 2, 2),
) -> dict:
    """
    Estimate parameter counts for each component of a DiT model
    without actually instantiating it.
    """
    patch_vol = math.prod(patch_size)

    # patch_embedding: Conv3d(in_dim -> dim, kernel=patch_size)
    p_patch = in_dim * dim * patch_vol + dim  # weight + bias

    # text_embedding: Linear(text_dim, dim) + Linear(dim, dim)
    p_text = (text_dim * dim + dim) + (dim * dim + dim)

    # time_embedding: Linear(freq_dim, dim) + Linear(dim, dim)
    p_time = (freq_dim * dim + dim) + (dim * dim + dim)

    # time_projection: Linear(dim, dim*6)
    p_time_proj = dim * (dim * 6) + (dim * 6)

    # Per block: self_attn + cross_attn + ffn + norms + modulation
    # Self-attn: Q, K, V, O (each dim->dim) + 2 x RMSNorm(dim)
    p_self_attn = 4 * (dim * dim + dim) + 2 * dim  # 2 RMSNorm weights
    # Cross-attn: same structure
    p_cross_attn = 4 * (dim * dim + dim) + 2 * dim
    # FFN: Linear(dim, ffn_dim) + Linear(ffn_dim, dim)
    p_ffn = (dim * ffn_dim + ffn_dim) + (ffn_dim * dim + dim)
    # LayerNorms: norm1(no affine), norm2(no affine), norm3(affine)
    p_norms = dim + dim  # norm3 has weight + bias (elementwise_affine=True)
    # Modulation: (1, 6, dim)
    p_mod = 6 * dim

    p_per_block = p_self_attn + p_cross_attn + p_ffn + p_norms + p_mod
    p_blocks = p_per_block * num_layers

    # Head: LayerNorm(dim, no affine) + Linear(dim, out_dim * patch_vol) + modulation(1, 2, dim)
    p_head = (dim * (out_dim * patch_vol) + out_dim * patch_vol) + 2 * dim

    total = p_patch + p_text + p_time + p_time_proj + p_blocks + p_head

    return {
        "patch_embedding": p_patch,
        "text_embedding":  p_text,
        "time_embedding":  p_time,
        "time_projection": p_time_proj,
        f"blocks (x{num_layers})": p_blocks,
        "head":            p_head,
        "TOTAL":           total,
    }


# ---- Define the three model configs ----
configs = {
    "Nano (~1.5M)": dict(
        dim=128, ffn_dim=512, num_layers=2, num_heads=4,
        in_dim=4, out_dim=4, text_dim=64, freq_dim=64,
    ),
    "Wan 1.3B": dict(
        dim=1536, ffn_dim=8960, num_layers=30, num_heads=12,
        in_dim=16, out_dim=16, text_dim=4096, freq_dim=256,
    ),
    "Wan 14B": dict(
        dim=5120, ffn_dim=13824, num_layers=40, num_heads=40,
        in_dim=16, out_dim=16, text_dim=4096, freq_dim=256,
    ),
}

# ---- Compute and display ----
print(f"{'Component':<22}", end="")
for name in configs:
    print(f" {name:>18}", end="")
print()
print("-" * (22 + 18 * len(configs) + len(configs)))

results = {name: estimate_params(**cfg) for name, cfg in configs.items()}

# Print each component row
for component in list(results["Nano (~1.5M)"].keys()):
    print(f"{component:<22}", end="")
    for name in configs:
        val = results[name].get(component, 0)
        if val >= 1e9:
            print(f" {val / 1e9:>15.2f} B", end="")
        elif val >= 1e6:
            print(f" {val / 1e6:>15.2f} M", end="")
        elif val >= 1e3:
            print(f" {val / 1e3:>15.1f} K", end="")
        else:
            print(f" {val:>17,}", end="")
    print()

# ---- Scaling ratios ----
nano_total = results["Nano (~1.5M)"]["TOTAL"]
print("\n--- Scaling Ratios (relative to Nano) ---")
for name in configs:
    t = results[name]["TOTAL"]
    print(f"  {name}: {t / nano_total:,.0f}x  ({t:,.0f} params)")

print(f"\nKey insight: Going from dim=128 to dim=5120 is a {5120/128:.0f}x width increase.")
print(f"Since most parameters scale as O(dim^2), this alone gives ~{(5120/128)**2:.0f}x more params.")
print(f"Combined with 40 layers (vs 2), the total scaling is ~{results['Wan 14B']['TOTAL'] / nano_total:,.0f}x.")

## Summary

In this notebook we have seen that:

1. **DiT replaces U-Net** with a Transformer backbone.  The key innovations are timestep modulation via AdaIN, bidirectional (no causal mask) self-attention, cross-attention for text conditioning, and 3D RoPE for spatial-temporal position encoding.

2. **NanoDiT is a faithful miniature** of Wan 14B.  Every component type is preserved -- only the width (`dim`), depth (`num_layers`), and number of heads differ.

3. **Scaling is straightforward**: the same architecture, made wider and deeper, accounts for the jump from ~1.5 M to ~14 B parameters.  The dominant scaling factor is `dim` because attention and FFN layers have $O(\text{dim}^2)$ parameters.

4. **Most capacity lives in the transformer blocks** (self-attention, cross-attention, and FFN), with a smaller but important role for the time and text embeddings.

Next, we will see how the **flow matching** diffusion process trains and uses this model to generate video.